# 02 — Feature Engineering V2

**Objetivo:** Transformar el dataset limpio en un dataset listo para modelado utilizando técnicas modernas de encoding.

## 1. Eliminar features sin valor predictivo (Data Leakage)
Igual que en V1, descartamos `anio_creacion` (varianza cero en 2025) y `subtipo_interes` (información post-cualificación). 

## 2. Crear nuevas features derivadas
Creamos `es_fin_de_semana` y `franja_horaria` basándonos en los hallazgos del EDA sobre leads nocturnos y de fin de semana.

## 3. Agrupar categorías de baja frecuencia
Para evitar el sobreajuste, agrupamos categorías con < 1% de presencia en la etiqueta "otros".

## 4. Split Train/Test Estratificado
Dividimos el dataset (80/20) antes de cualquier encoding para evitar que el modelo "vea" el futuro.

In [ ]:
import pandas as pd
import numpy as np
from category_encoders import TargetEncoder
from sklearn.model_selection import train_test_split

df = pd.read_csv("../data/processed/leads_cleaned.csv")
df = df.drop(columns=["subtipo_interes"])

# Features derivadas
df["es_fin_de_semana"] = df["dia_semana_creacion"].isin(["sábado", "domingo"]).astype(int)
def clasificar_franja(hora):
    if 0 <= hora < 6: return "madrugada"
    elif 6 <= hora < 12: return "manana"
    elif 12 <= hora < 18: return "tarde"
    return "noche"
df["franja_horaria"] = df["hora_creacion"].apply(clasificar_franja)

# Split
X = df.drop(columns=["target"])
y = df["target"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f"Train: {X_train.shape[0]} | Test: {X_test.shape[0]}")

## 5. Bayesian Target Encoding (Reemplaza al método manual de V1)

### 5.1 ¿Por qué ya no usamos el "Encoding de contribución" manual?
En la **V1**, creábamos manualmente dos columnas (`_contribucion` y `_tasa_suavizada`) aplicando una fórmula de suavizado para evitar que categorías raras con 100% de conversión engañaran al modelo.

En la **V2**, utilizamos la librería oficial `TargetEncoder(smoothing=100)`. Esta aplica el **Suavizado Bayesiano** de forma automática y matemáticamente robusta:
1.  **Reduce la dimensionalidad:** Pasamos de 48 columnas (One-Hot) a solo 11 features densas.
2.  **Evita el Overfitting:** Penaliza automáticamente a los concesionarios o campañas con pocos leads.
3.  **Mayor Poder Predictivo:** Captura la probabilidad real de conversión en una sola variable numérica.

In [ ]:
# Aplicando el encoding moderno
cat_cols = ["vehiculo_interes", "origen", "nombre_formulario", "campana", "concesionario"]
encoder = TargetEncoder(cols=cat_cols, smoothing=100)

X_train_enc = encoder.fit_transform(X_train, y_train)
X_test_enc = encoder.transform(X_test)

# One-Hot para el resto de baja cardinalidad
X_train_final = pd.get_dummies(X_train_enc, drop_first=True, dtype=int)
X_test_final = pd.get_dummies(X_test_enc, drop_first=True, dtype=int).reindex(columns=X_train_final.columns, fill_value=0)

print(f"Features finales: {X_train_final.shape[1]} (V1 tenía 48)")

In [ ]:
# Exportar
X_train_final.to_csv("../data/processed/X_train_v2.csv", index=False)
X_test_final.to_csv("../data/processed/X_test_v2.csv", index=False)
y_train.to_csv("../data/processed/y_train_v2.csv", index=False)
y_test.to_csv("../data/processed/y_test_v2.csv", index=False)